# Prepare the environment for the notebook

In [ ]:
# sklearn, joblib, s3fs already come with the notebook image
# %pip install sklearn joblib s3fs

# Create a small model to be deployed as InferenceService

In [ ]:
from sklearn import svm, datasets
from joblib import dump

In [ ]:
# Create a small model with iris dataset
iris = datasets.load_iris()
clf = svm.SVC(gamma='scale')
clf.fit(iris.data, iris.target)
dump(clf, 'model.joblib')
print("Iris model file 'model.joblib' created!")

# Push the created model to s3 storage (MinIO)

The code cell below automatically sets the bucket for this experiment to `<namespace>-data` (bucket available by default in prokube deployments).

<div style="
  border: 1px solid #cce5ff;
  background-color: #eaf4ff;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
">
  <strong>💡 Tip:</strong>
  <div style="margin-top: 6px;">
    If you want to use another bucket, modify the <code>s3_bucket</code> variable in the cell below.
  </div>
</div>

In [ ]:
with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace", "r") as namespace_file:
    namespace = namespace_file.read()
s3_bucket = f"{namespace}-data"
s3_model_path = f"{s3_bucket}/minimal-kserve-example"
print(f"The created model will be uploaded to s3://{s3_model_path}")

Initialize s3 client:

In [ ]:
import s3fs
s3 = s3fs.S3FileSystem() # Reads the s3 credentials and endpoint from the environment variables

# If you want to use a different s3 instance, make sure to set 
# s3fs.S3FileSystem(endpoint_url=<endpoint_url>, key=<key>, secret=<secret>)

Upload the model to MinIO:

In [ ]:
s3.put("model.joblib", f"{s3_model_path}/model.joblib")
# List the bucket content to see if upload was successful
s3.ls(s3_model_path)

# Create the InferenceService manifest that will use the uploaded model and deploy it to the cluster

Create and write the manifest for the KServe InferenceService:


In [ ]:
inference_service_name = "kserve-minio-test"
inference_service_manifest= \
f"""
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: {inference_service_name}
spec:
  predictor:
    model:
      modelFormat:
        name: sklearn
      storageUri: s3://{s3_model_path}/model.joblib

"""
manifest_file_name = "inferenceservice.yaml"
with open(manifest_file_name, "w") as manifest_file:
    manifest_file.write(inference_service_manifest)
print(f"InferenceService manifest saved to ./{manifest_file_name}")

Use `kubectl` to deploy the created manifest:

In [ ]:
# Jupyter notebook replaces {variable} with actual python value
!kubectl apply -f {manifest_file_name}

Wait for the KServe InferenceService to be ready using `kubectl`:

In [ ]:
!kubectl wait inferenceservice --for=condition=ready --timeout 300s {inference_service_name} && echo
# Get InferenceService status
!kubectl get inferenceservice {inference_service_name} \
  -o 'custom-columns=NAME:.metadata.name,URL:.status.url,READY:.status.conditions[?(@.type=="Ready")].status'

<div style="
  border: 1px solid #cce5ff;
  background-color: #eaf4ff;
  padding: 12px 16px;
  border-radius: 8px;
  margin: 12px 0;
">
  <span style="font-size: 18px;">💡</span>
  <strong> Tip</strong>
  <div style="margin-top: 6px;">
    To watch the model in the UI, go to the prokube central dashboard / prokube UI, 
    open <b>Endpoints</b> (or <b>Models</b>), and search for the model name.
  </div>
</div>

# Test the deployed InferenceService with a sample request

Test using the internal URL:

In [ ]:
import requests
isvc_internal_url = f"{inference_service_name}-predictor.{namespace}"  # KServe internal endpoint is always built like this
response = requests.post(
    f"http://{isvc_internal_url}/v1/models/{inference_service_name}:predict",
    # an iris instance is [sepal_length, sepal_width, petal_length, petal_width]
    json={"instances": [[6.8, 2.8, 4.8, 1.4], [5.1, 3.5, 1.4, 0.2]]}
)
print(response.json())

## Test using external URL

Enter the model serving API key in the cell below. If not known, ask the cluster administrator for it.

In [ ]:
INFERENCE_SERVICE_API_KEY = ""
if not INFERENCE_SERVICE_API_KEY:
    raise RuntimeError("Please provide the API Key that will be used to test the deployed InferenceService")

Obtain the InferenceService's external URL from KServe:

In [ ]:
# Below, we use {{ and }} to escape the curly braces in the jsonpath expression 
# so Jupyter notebook does not try to replace them with python variables.
isvc_external_url = !kubectl get inferenceservice {inference_service_name} -o jsonpath='{{.status.url}}' 
isvc_external_url = isvc_external_url[0]  # Jupyter notebook shell command executions returns an array
print(isvc_external_url)

In [ ]:
import requests
response = requests.post(
    f"{isvc_external_url}/v1/models/{inference_service_name}:predict",
    headers={
        "X-Api-Key": INFERENCE_SERVICE_API_KEY
    },
    json={"instances": [[6.8, 2.8, 4.8, 1.4], [5.1, 3.5, 1.4, 0.2]]}
)
print(response.json())